In [ ]:
import os
import random
from dataclasses import dataclass, field
from typing import Optional, Tuple, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal, Bernoulli
from torch.optim import AdamW
from datasets import load_dataset, Dataset, concatenate_datasets
from transformers import HfArgumentParser, AutoTokenizer
from collections import defaultdict
import numpy as np
from downstream_utils import PetsPreferenceEnv, evaluate


class HighLevelPolicy(nn.Module):
    def __init__(self, pretrained_vae, tokenizer_name="gpt2", device=None):
        super().__init__()
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.vae = pretrained_vae

        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.tokenizer.pad_token_id = self.tokenizer.eos_token_id
        self.tokenizer.padding_side = "right"

        latent_dim = self.vae.latent_dim

        self.pair_encoder = self.vae.pair_encoder

        pair_param = next(self.pair_encoder.parameters())
        self.model_device = pair_param.device
        self.model_dtype = pair_param.dtype

        self.mean_head = nn.Linear(latent_dim, latent_dim).to(
            device=self.model_device, dtype=self.model_dtype
        )
        self.logvar_head = nn.Linear(latent_dim, latent_dim).to(
            device=self.model_device, dtype=self.model_dtype
        )

        self.to(self.model_device)

    @torch.no_grad()
    def _embed_text(self, text: str) -> torch.Tensor:
        tokens = self.tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
        )
        tokens = {k: v.to(self.model_device) for k, v in tokens.items()}

        emb = self.vae.llm_encoder(
            input_ids=tokens["input_ids"],
            attention_mask=tokens["attention_mask"],
        )[0]

        if emb.dim() == 1:
            emb = emb.unsqueeze(0)

        emb = emb.to(device=self.model_device, dtype=self.model_dtype)
        return emb

    def forward(self, x: str, y: str):
        with torch.no_grad():
            x_emb = self._embed_text(x)
            y_emb = self._embed_text(y)

        x_emb = x_emb.to(device=self.model_device, dtype=self.model_dtype)
        y_emb = y_emb.to(device=self.model_device, dtype=self.model_dtype)

        pair_hidden = self.pair_encoder(x_emb, y_emb)
        pair_hidden = pair_hidden.to(device=self.model_device, dtype=self.model_dtype)

        mean = torch.clamp(self.mean_head(pair_hidden), -3, 3)
        log_var = torch.clamp(self.logvar_head(pair_hidden), -1, 1)
        std = torch.exp(0.5 * log_var)

        dist = Normal(mean, std)
        z = dist.rsample()
        z = z.to(device=self.model_device, dtype=self.model_dtype)

        return {
            "x_emb": x_emb,
            "y_emb": y_emb,
            "mean": mean,
            "log_var": log_var,
            "z": z,
            "dist": dist,
        }


class FrozenLowLevelPolicy(nn.Module):
    def __init__(self, pretrained_vae, device=None):
        super().__init__()
        self.vae = pretrained_vae
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")

        decoder_param = next(self.vae.decoder.parameters())
        self.decoder_device = decoder_param.device
        self.decoder_dtype = decoder_param.dtype

        # freeze decoder
        for p in self.vae.decoder.parameters():
            p.requires_grad = False
        self.vae.decoder.eval()

    def score(self, x_emb, y_emb, z):
        x_emb = x_emb.to(device=self.decoder_device, dtype=self.decoder_dtype)
        y_emb = y_emb.to(device=self.decoder_device, dtype=self.decoder_dtype)
        z = z.to(device=self.decoder_device, dtype=self.decoder_dtype)

        score_x, score_y = self.vae.decode(x_emb, y_emb, z)
        return score_x, score_y


class BanditVAEPolicy(nn.Module):
    def __init__(self, vae_model_path: str, tokenizer_name="gpt2", device=None, temperature=1.0):
        super().__init__()
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.vae = torch.load(vae_model_path, map_location=self.device, weights_only=False)
        self.vae.to(self.device)
        self.vae.eval()

        # freeze all pretrained modules first
        for p in self.vae.parameters():
            p.requires_grad = False

        self.high_level = HighLevelPolicy(self.vae, tokenizer_name=tokenizer_name, device=self.device)
        self.low_level = FrozenLowLevelPolicy(self.vae, device=self.device)
        self.temperature = temperature
        for p in self.high_level.pair_encoder.parameters():
            p.requires_grad = True
        for p in self.high_level.mean_head.parameters():
            p.requires_grad = True
        for p in self.high_level.logvar_head.parameters():
            p.requires_grad = True

    def act(self, pair: Tuple[str, str], sample=True):
        x, y = pair
        out = self.high_level(x, y)

        x_emb, y_emb, z = out["x_emb"], out["y_emb"], out["z"]

        decoder_param = next(self.vae.decoder.parameters())
        decoder_device = decoder_param.device
        decoder_dtype = decoder_param.dtype

        x_emb = x_emb.to(device=decoder_device, dtype=decoder_dtype)
        y_emb = y_emb.to(device=decoder_device, dtype=decoder_dtype)
        z = z.to(device=decoder_device, dtype=decoder_dtype)

        score_x, score_y = self.low_level.score(x_emb, y_emb, z)
        logits = (score_x - score_y) / self.temperature
        action_dist = Bernoulli(logits=logits)

        if sample:
            action = action_dist.sample()
        else:
            action = (torch.sigmoid(logits) > 0.5).to(logits.dtype)

        return {
            **out,
            "score_x": score_x,
            "score_y": score_y,
            "logits": logits,
            "action_dist": action_dist,
            "action": int(action.item()),
            "action_tensor": action,
        }



@dataclass
class ScriptArguments:
    vae_model_path: str = field(default="logs/gpt2_simple_pets/both/vae_gpt2__0_0.0003_cosine_2_0.0001_512_768_seed0_peft_last_checkpoint/model.pt")
    tokenizer_name: str = field(default="gpt2")
    data_path: str = field(default="data/simple_pets/gpt2")
    data_subset: str = field(default="both")
    split: str = field(default="train")
    learning_rate: float = field(default=1e-4)
    num_steps: int = field(default=5000)
    kl_coef: float = field(default=0.005)
    entropy_coef: float = field(default=1e-1)
    baseline_momentum: float = field(default=0.9)
    temperature: float = field(default=1.0)
    seed: int = field(default=0)
    eval_every: int = field(default=200)
    save_dir: str = field(default="logs/vae_bandit_policy")



In [ ]:
args = ScriptArguments()

random.seed(args.seed)
torch.manual_seed(args.seed)

device = "cuda" if torch.cuda.is_available() else "cpu"



In [ ]:
from collections import defaultdict
import numpy as np
import os
import random
import torch
from torch.optim import AdamW

def run(env_id, seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    policy = BanditVAEPolicy(
        vae_model_path=args.vae_model_path,
        tokenizer_name=args.tokenizer_name,
        device=device,
        temperature=args.temperature,
    )

    # optimizer: only high-level trainable params
    trainable_params = [p for p in policy.parameters() if p.requires_grad]
    optimizer = AdamW(trainable_params, lr=args.learning_rate)

    train_env = PetsPreferenceEnv(
        seed=seed, pref_type=env_id
    )
    test_env = PetsPreferenceEnv(
        seed=seed + 1, pref_type=env_id, eval=True
    )

    reward_baseline = 0.0
    state = train_env.reset()

    os.makedirs(args.save_dir, exist_ok=True)
    best_acc = -1.0
    final_acc, final_cost = None, None

    for step in range(1, args.num_steps + 1):
        out = policy.act(state, sample=True)
        next_state, reward, done, info = train_env.step(out["action"])

        reward_t = torch.tensor(float(reward), device=device)

        reward_baseline = (
            args.baseline_momentum * reward_baseline
            + (1 - args.baseline_momentum) * reward
        )
        advantage = reward_t - reward_baseline

        log_prob_action = out["action_dist"].log_prob(out["action_tensor"]).mean()

        log_prob_z = out["dist"].log_prob(out["z"]).sum(dim=-1).mean()

        entropy = out["action_dist"].entropy().mean()

        mean = out["mean"]
        log_var = out["log_var"]
        kld = -0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp(), dim=-1).mean()

        pg_loss = -(advantage.detach() * log_prob_action)
        latent_loss = -50.0 * log_prob_z
        loss = pg_loss + latent_loss - args.entropy_coef * entropy  # + args.kl_coef * kld

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
        optimizer.step()

        state = next_state

        if step % 50 == 0:
            print(
                f"seed={seed} env_id={env_id} step={step} "
                f"reward={reward:.3f} "
                f"baseline={reward_baseline:.3f} "
                f"adv={float(advantage):.3f} "
                f"loss={float(loss):.4f} "
                f"kld={float(kld):.4f} "
                f"entropy={float(entropy):.4f} "
                f"mean={mean.mean().item():.4f} "
                f"log_var={log_var.mean().item():.4f}"
            )

        if step % args.eval_every == 0 or step == args.num_steps:
            res = evaluate(policy, test_env, num_episodes=600)
            acc = res["overall_accuracy"]
            avg_cost = res["overall_avg_cost"]
            print(
                f"[eval] seed={seed} env_id={env_id} step={step}, "
                f"test_acc={acc:.4f}, test_avg_cost={avg_cost:.4f}"
            )

            if step == args.num_steps:
                final_acc, final_cost = acc, avg_cost

            if acc > best_acc:
                best_acc = acc
                ckpt_path = os.path.join(args.save_dir, f"best_high_level_seed{seed}_env{env_id}.pt")
                torch.save(
                    {
                        "pair_encoder": policy.high_level.pair_encoder.state_dict(),
                        "mean_head": policy.high_level.mean_head.state_dict(),
                        "logvar_head": policy.high_level.logvar_head.state_dict(),
                        "step": step,
                        "best_acc": best_acc,
                        "seed": seed,
                        "env_id": env_id,
                    },
                    ckpt_path,
                )
                print(f"saved best checkpoint to {ckpt_path}")

    return {
        "acc": float(final_acc),
        "cost": float(final_cost),
    }


def summarize_results(all_results):
    seed_avgs_acc = [x["avg_acc"] for x in all_results]
    seed_avgs_cost = [x["avg_cost"] for x in all_results]

    mean_acc = float(np.mean(seed_avgs_acc))
    std_acc = float(np.std(seed_avgs_acc))
    mean_cost = float(np.mean(seed_avgs_cost))
    std_cost = float(np.std(seed_avgs_cost))

    print("\n" + "=" * 80)
    print("Ours")
    print(f"Across 3 seeds:")
    print(f"  avg_reward = {mean_acc:.4f} +- {std_acc:.4f}")
    print(f"  avg_cost   = {mean_cost:.4f} +- {std_cost:.4f}")
    print("=" * 80)


seeds = [0, 1, 2]
all_results = []

for seed in seeds:
    env0_res = run(env_id=0, seed=seed)
    env1_res = run(env_id=1, seed=seed)

    avg_acc = 0.5 * (env0_res["acc"] + env1_res["acc"])
    avg_cost = 0.5 * (env0_res["cost"] + env1_res["cost"])

    all_results.append({
        "seed": seed,
        "env0": env0_res,
        "env1": env1_res,
        "avg_acc": avg_acc,
        "avg_cost": avg_cost,
    })

    print(
        f"[seed summary] seed={seed}: "
        f"env0(acc={env0_res['acc']:.4f}, cost={env0_res['cost']:.4f}), "
        f"env1(acc={env1_res['acc']:.4f}, cost={env1_res['cost']:.4f}), "
        f"avg_reward={avg_acc:.4f}, avg_cost={avg_cost:.4f}"
    )


In [4]:
summarize_results(all_results)


Ours
Across 3 seeds:
  avg_reward = 0.7500 +- 0.0000
  avg_cost   = 0.0000 +- 0.0000
